In [19]:
## Matrix series indicator

import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
import numpy as np
import vectorbt as vbt
import pandas as pd
import numba as nb
import os

In [20]:
REMOTE_HOST = 'http://192.168.1.3:9000'

def load_stock_data() -> pd.DataFrame:
    from deltalake import DeltaTable
    storage_options = {
        'AWS_ACCESS_KEY_ID':          'CzOwnLkEDXQy951AOqes',
        'AWS_SECRET_ACCESS_KEY':      'fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S',
        'AWS_ENDPOINT_URL':           REMOTE_HOST,
        'AWS_ALLOW_HTTP':             'true',
        'AWS_EC2_METADATA_DISABLED':  'true',
        'AWS_REGION':                 'us-east-1',
        'aws_conditional_put':        'etag',
    }
    watchlist_df      = pd.read_csv('../backend/models/watchlist.csv')
    watchlist_symbols = watchlist_df.iloc[:, 0].values
    start_date        = pd.Timestamp.now() - pd.DateOffset(years=10)

    dt = DeltaTable('s3://delta-table-storage/stocks', storage_options=storage_options)
    raw = dt.to_pandas(
        filters=[('date', '>=', start_date), ('symbol', 'in', watchlist_symbols)],
        columns=['symbol', 'date', 'close', 'open', 'high', 'low', 'volume'],
    )
    raw = raw.drop_duplicates(subset=['date', 'symbol'], keep='last')
    raw = raw.set_index(['date', 'symbol'])
    stocks = raw.unstack(level=1).bfill().ffill()
    print('Loaded from DeltaLake')
    return stocks


local_file = 'stocks_data_latest.h5'
store_key  = 'stocks'

if os.path.exists(local_file):
    print(f'Loading from local HDF5: {local_file}')
    with pd.HDFStore(local_file, mode='r') as store:
        df_raw = store[store_key]
else:
    print('HDF5 not found — loading from DeltaLake …')
    df_raw = load_stock_data()
    with pd.HDFStore(local_file, mode='w') as store:
        store.put(store_key, df_raw)

watchlist_df      = pd.read_csv('../backend/models/watchlist.csv')
watchlist_symbols = watchlist_df.iloc[:, 0].values
df_raw = df_raw.loc[:, df_raw.columns.get_level_values('symbol').isin(watchlist_symbols)]

# ── extract OHLCV DataFrames (dates × symbols) ────────────────────────────────
open_   = df_raw['open']
high    = df_raw['high']
low     = df_raw['low']
close   = df_raw['close']
volume  = df_raw['volume']

print(f'Shape: {close.shape}  |  {close.index[0]} → {close.index[-1]}')
close.tail(3)

Loading from local HDF5: stocks_data_latest.h5
Shape: (2501, 198)  |  2016-06-20 00:00:00 → 2026-06-19 00:00:00


symbol,AAA,ABB,ACB,ACV,ADS,AGG,AGR,ANV,APH,ASM,...,VNINDEX,VNM,VOS,VPB,VPG,VPI,VRE,VSC,VTP,YEG
date,,,,,,,,,,,,,,,,,,,,,
2026-06-17,7.40,16.9,22.0,44.6,9.48,12.05,15.00,21.90,5.69,5.93,...,1806.20,59.0,12.85,26.5,2.93,60.5,28.15,20.25,65.5,8.88
2026-06-18,7.38,17.2,22.4,44.5,9.43,11.90,14.75,21.80,5.61,5.90,...,1830.47,59.2,12.65,26.4,2.90,60.0,30.10,19.90,65.5,8.73
2026-06-19,7.43,17.4,22.2,44.2,9.38,11.90,15.05,21.45,5.54,5.87,...,1824.53,59.0,12.65,25.9,2.89,58.9,29.35,19.20,65.0,8.64


In [ ]:
pd.set_option('display.max_rows', 200)

def wilders(s: pd.Series, n: int) -> pd.Series:
    """AFL Wilders smoothing ~ EMA with alpha=1/n, adjust=False."""
    return s.ewm(alpha=1 / n, adjust=False).mean()

def wilders_ndarray(s: np.ndarray, n: int) -> np.ndarray:
    rows, cols = s.shape
    result = np.zeros_like(s)
    for col in range(cols):
        # pandas ewm can't be used on numpy array, so use pandas.Series per column:
        result[:, col] = pd.Series(s[:, col]).ewm(alpha=1 / n, adjust=False).mean().values
    return result

def matrix_series(close: np.ndarray, high: np.ndarray, low: np.ndarray, price_period: int = 20, supResPeriod: int = 50, supResPercentage: int = 100, smoother: int = 5):
  
    Osc = vbt.IndicatorFactory.from_talib('CCI').run(high, low, close, timeperiod=price_period).real

    Value1 = Osc
    Value2 = vbt.IndicatorFactory.from_talib('MAX').run(Value1, timeperiod=supResPeriod).real
    Value3 = vbt.IndicatorFactory.from_talib('MIN').run(Value1, timeperiod=supResPeriod).real
    Value4 = Value2 - Value3
    Value5 = Value4 * (supResPercentage / 100.0)

    ResistanceLine = Value3 + Value5
    SupportLine = Value2 - Value5

    # Signal line 
    ys1 = (high + low + close * 2) / 4.0
    # rk3 = vbt.MA.run(ys1, window=smoother, ewm=True).ma
    rk3 = vbt.IndicatorFactory.from_talib('EMA').run(ys1, timeperiod=smoother).real.to_numpy()
    rk4 = vbt.IndicatorFactory.from_talib('STDDEV').run(ys1, timeperiod=smoother, nbdev=1).real.to_numpy()
    rk5 = (ys1 - rk3) * 200.0 / rk4

    # rk6 = vbt.MA.run(rk5, window=smoother, ewm=True).ma
    rk6 = vbt.IndicatorFactory.from_talib('EMA').run(rk5, timeperiod=smoother).real
    UP_line = vbt.IndicatorFactory.from_talib('EMA').run(rk6, timeperiod=smoother).real
    DOWN_line = vbt.IndicatorFactory.from_talib('EMA').run(UP_line, timeperiod=smoother).real

    # Candle OHLC
    Hh = UP_line.where(UP_line < DOWN_line, DOWN_line)      # High = min(up, down)
    Ll = DOWN_line.where(UP_line < DOWN_line, UP_line)      # Low = max(up, down)  
    
    return Hh, Ll, SupportLine, ResistanceLine

## Matrix series indicator
matrix_series_indicator = vbt.IndicatorFactory(
    class_name='MatrixSeries',
    short_name='matrix_series',
    input_names=['close', 'high', 'low'],
    param_names=['price_period', 'supResPeriod', 'supResPercentage', 'smoother'],
    output_names=['hh', 'll', 'support_line', 'resistance_line']
).from_apply_func(matrix_series)

matrix_series = matrix_series_indicator.run(df.close.round(2), df.high.round(2), df.low.round(2), 
    price_period=20, supResPeriod=50, supResPercentage=100, smoother=5)

up = matrix_series.hh
down = matrix_series.ll
support_line = matrix_series.support_line
resistance_line = matrix_series.resistance_line

resistance_line.tail(100)

In [34]:
## Bollinger Band Width (BBW) Squeeze Scanner
# BBW = (Upper - Lower) / Middle  → normalized band width
# BBW percentile over rolling window → 0 = most squeezed (coiled), 100 = widest

from numpy.lib.stride_tricks import sliding_window_view


def rolling_pct_rank(bbw: pd.DataFrame, window: int) -> pd.DataFrame:
    """Rolling percentile rank of the current bar within its window (matches pandas rank pct=True)."""
    min_periods = window // 2
    arr = bbw.to_numpy(dtype=np.float64)
    n_rows, n_cols = arr.shape
    out = np.full((n_rows, n_cols), np.nan)

    for j in range(n_cols):
        col = arr[:, j]
        if n_rows < window:
            continue
        sw = sliding_window_view(col, window)          # (n - w + 1, w)
        cur = col[window - 1:]
        finite = np.isfinite(sw)
        valid = finite.sum(axis=1)
        ok = valid >= min_periods

        less = ((sw < cur[:, None]) & finite).sum(axis=1)
        equal = ((sw == cur[:, None]) & finite).sum(axis=1)
        rank = less + (equal + 1.0) / 2.0              # average rank, 1-based

        pct = np.full(sw.shape[0], np.nan)
        pct[ok] = rank[ok] / valid[ok] * 100.0
        out[window - 1:, j] = pct

    return pd.DataFrame(out, index=bbw.index, columns=bbw.columns)


def bbw_squeeze(close, bb_period: int = 20, bb_std: float = 2.0, pct_window: int = 60) -> tuple:
    """
    Returns:
        bbw        : raw band width series
        bbw_pct    : rolling percentile [0-100], lower = tighter squeeze
    """
    bb = vbt.IndicatorFactory.from_talib('BBANDS').run(
        close, timeperiod=bb_period, nbdevup=bb_std, nbdevdn=bb_std, matype=0
    )
    bbw = (bb.upperband - bb.lowerband) / bb.middleband
    bbw_pct = rolling_pct_rank(bbw, pct_window)
    return bbw, bbw_pct


bbw, bbw_pct = bbw_squeeze(close, bb_period=14, bb_std=1.4, pct_window=60)

# Snapshot: latest BBW and percentile rank per symbol, sorted ascending (most squeezed first)
latest = pd.DataFrame({
    'bbw':     bbw.iloc[-1].round(4),
    'bbw_pct': bbw_pct.iloc[-1].round(1),
}).sort_values('bbw_pct')

print("=== BBW Squeeze Ranking (lower pct = more coiled) ===")
# latest

## pick a date
date = '2026-06-18'
picked_data = pd.DataFrame({
    'bbw':     bbw.loc[date].round(4),
    'bbw_pct': bbw_pct.loc[date].round(1),
}).sort_values('bbw_pct')
picked_data

=== BBW Squeeze Ranking (lower pct = more coiled) ===


bbw  \
bbands_timeperiod bbands_nbdevup bbands_nbdevdn bbands_matype symbol            
14                1.4            1.4            0             VCG      0.0401   
                                                              NT2      0.0311   
                                                              HTN      0.0444   
                                                              NVL      0.0931   
                                                              SCR      0.0485   
                                                              HDG      0.0263   
                                                              HDB      0.0222   
                                                              PLC      0.0353   
                                                              HUT      0.0146   
                                                              VCB      0.0122   
                                                              VPG      0.0763   
                                                              VN30     0.0220   
                                                              EVG      0.0338   
                                                              VPB      0.0345   
                                                              HNG      0.0222   
                                                              GIL      0.0377   
                                                              DXS      0.0481   
                                                              TCH      0.0389   
                                                              BAF      0.0228   
                                                              PVI      0.0103   
                                                              TIG      0.0329   
                                                              KHG      0.0174   
                                                              DRI      0.0424   
                                                              GEG      0.0228   
                                                              CNG      0.0170   
                                                              DDV      0.0210   
                                                              TV2      0.0459   
                                                              VTP      0.0348   
                                                              TLG      0.0000   
                                                              DXG      0.0381   
                                                              REE      0.0198   
                                                              BFC      0.0421   
                                                              BCM      0.0369   
                                                              FOX      0.0416   
                                                              KSB      0.0200   
                                                              NTL      0.0348   
                                                              EIB      0.0348   
                                                              CTD      0.0358   
                                                              PVT      0.0357   
                                                              QTP      0.0146   
                                                              MSR      0.0434   
                                                              NHA      0.0428   
                                                              HBC      0.0144   
                                                              PVS      0.0338   
                                                              VNINDEX  0.0275   
                                                              DCM      0.0709   
                                                              IDC      0.0276   
                                                              SMC      0.0463   
                        

In [23]:
## ATR / Range Contraction: NR7, NR4, Inside Bars, ATR Percentile
# NR7 : narrowest range of last 7 bars  → compressed spring, imminent move
# NR4 : narrowest range of last 4 bars  → shorter lookback, more frequent
# inside: bar fully inside prior bar    → indecision, often precedes breakout
# atr_pct: rolling ATR percentile       → 0 = multi-week low in volatility

def flatten_vbt_columns(df: pd.DataFrame, symbols=None) -> pd.DataFrame:
    """vbt talib outputs MultiIndex columns; flatten to symbol-only."""
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(-1)
    if symbols is not None:
        out = out.reindex(columns=symbols)
    return out


def range_contraction(high: pd.DataFrame, low: pd.DataFrame, close: pd.DataFrame,
                      atr_period: int = 14, pct_window: int = 125) -> dict:
    symbols = close.columns
    bar_range = high - low

    # NR7 / NR4: True on the bar whose range is the rolling minimum (inclusive)
    nr7 = bar_range <= bar_range.rolling(7).min()
    nr4 = bar_range <= bar_range.rolling(4).min()

    # Inside bar: range entirely swallowed by the previous bar
    inside = (high < high.shift(1)) & (low > low.shift(1))

    # Consecutive inside-bar count (streak) — longer streak = tighter coil
    inside_np = inside.to_numpy(dtype=np.bool_)
    streak_np = np.zeros_like(inside_np, dtype=np.int64)
    for j in range(inside_np.shape[1]):
        run = 0
        for i in range(inside_np.shape[0]):
            if inside_np[i, j]:
                run += 1
                streak_np[i, j] = run
            else:
                run = 0
    inside_streak_df = pd.DataFrame(streak_np, index=inside.index, columns=inside.columns)

    # ATR via talib, then rolling percentile (reuses rolling_pct_rank from BBW cell)
    atr_raw = flatten_vbt_columns(
        vbt.IndicatorFactory.from_talib('ATR').run(
            high, low, close, timeperiod=atr_period
        ).real,
        symbols=symbols,
    )

    atr_pct = rolling_pct_rank(atr_raw, pct_window)

    return dict(
        bar_range=bar_range,
        nr4=nr4,
        nr7=nr7,
        inside=inside,
        inside_streak=inside_streak_df,
        atr=atr_raw,
        atr_pct=atr_pct,
    )


rc = range_contraction(high, low, close, atr_period=14, pct_window=125)

# ── Watchlist snapshot sorted by ATR percentile (most compressed first) ───────
snap = pd.DataFrame({
    'range':          rc['bar_range'].iloc[-1].round(3),
    'nr4':            rc['nr4'].iloc[-1],
    'nr7':            rc['nr7'].iloc[-1],
    'inside':         rc['inside'].iloc[-1],
    'inside_streak':  rc['inside_streak'].iloc[-1].astype(int),
    'atr':            rc['atr'].iloc[-1].round(3),
    'atr_pct':        rc['atr_pct'].iloc[-1].round(1),
}).sort_values('atr_pct')

print("=== Range Contraction Snapshot (lower atr_pct = more coiled) ===")
print(snap.to_string())


=== Range Contraction Snapshot (lower atr_pct = more coiled) ===
         range    nr4    nr7  inside  inside_streak     atr  atr_pct
symbol                                                              
YEG       0.15  False  False   False              0   0.169      0.8
DGC       0.85  False  False   False              0   1.113      0.8
HSG       0.20  False  False   False              0   0.267      0.8
MSH       0.35   True  False   False              0   0.487      0.8
DRC       0.10   True   True   False              0   0.202      0.8
MIG       0.20   True   True   False              0   0.382      0.8
VCI       0.50  False  False   False              0   0.642      0.8
DXG       0.35  False  False   False              0   0.385      0.8
VCG       0.30  False  False   False              0   0.461      0.8
EIB       0.25  False  False   False              0   0.431      0.8
SJS       0.80  False  False   False              0   1.236      0.8
MBB       0.25  False  False   False  

In [38]:
## BBW Squeeze Breakout Strategy — vectorbt
# ARM   : bbw_pct < squeeze_thresh for ≥ min_squeeze_bars, then crosses up while bbw rising
# DIRECT: while armed, close > Donchian high (long) | close < Donchian low (short)
#         volume gate: optional (set VOL_MULT=0 to skip)
# EXIT  : ATR trailing stop (atr_trailing_nb, same as 005d) + swing-low initial sl_stop

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from backend.app.services.indicators.trailing_sl import atr_trailing_nb

# ── params ────────────────────────────────────────────────────────────────────
SQUEEZE_THRESH   = 20     # bbw_pct below this = coiling
MIN_SQUEEZE_BARS = 10      # consecutive bars required below threshold
ARM_WINDOW       = 5      # bars the signal stays armed after squeeze fires
DONCHIAN_PERIOD  = 10     # Donchian high/low lookback
VOL_MULT         = 0    # volume > N × rolling mean; set 0 to disable
VOL_WINDOW       = 20
ATR_MULT         = 2.0    # ATR trailing-stop multiplier
ATR_PERIOD       = 14
SWING_LOW_LB     = 5      # swing-low lookback for initial hard stop
MAX_HOLD_DAYS    = 0     # time stop (0 = disabled)

# ── 1. ARM signal (reuses bbw / bbw_pct from BBW cell) ───────────────────────
below_thresh = bbw_pct < SQUEEZE_THRESH

below_np  = below_thresh.to_numpy(dtype=np.bool_)
consec_np = np.zeros(below_np.shape, dtype=np.int32)
for j in range(below_np.shape[1]):
    run = 0
    for i in range(below_np.shape[0]):
        run = run + 1 if below_np[i, j] else 0
        consec_np[i, j] = run
consec_df = pd.DataFrame(consec_np, index=close.index, columns=close.columns)

bbw_rising    = bbw > bbw.shift(1)
cross_up      = (~below_thresh) & below_thresh.shift(1).fillna(False)
qualified     = consec_df.shift(1).fillna(0) >= MIN_SQUEEZE_BARS
squeeze_fired = cross_up & bbw_rising & qualified

armed_np = squeeze_fired.to_numpy(dtype=np.bool_)
arm_out  = np.zeros(armed_np.shape, dtype=np.int32)
for j in range(armed_np.shape[1]):
    ttl = 0
    for i in range(armed_np.shape[0]):
        if armed_np[i, j]:
            ttl = ARM_WINDOW
        arm_out[i, j] = ttl
        if ttl > 0:
            ttl -= 1
armed_df = pd.DataFrame(arm_out, index=close.index, columns=close.columns) > 0

# ── 2. DIRECTION — Donchian breakout (shift(1) = no lookahead) ───────────────
don_high = high.shift(1).rolling(DONCHIAN_PERIOD).max()
don_low  = low.shift(1).rolling(DONCHIAN_PERIOD).min()

if VOL_MULT > 0:
    vol_ok = volume > VOL_MULT * volume.rolling(VOL_WINDOW).mean()
else:
    vol_ok = pd.DataFrame(True, index=close.index, columns=close.columns)

entries = armed_df & (close > don_high) & vol_ok   # long only

# ── 3. EXIT — ATR trailing stop (same as 005d compute_exits) ─────────────────
_MIN = vbt.IndicatorFactory.from_talib('MIN')
_ATR = vbt.IndicatorFactory.from_talib('ATR')

atr_raw = _ATR.run(high, low, close, timeperiod=ATR_PERIOD).real
# flatten vbt MultiIndex → symbol-only columns
if isinstance(atr_raw.columns, pd.MultiIndex):
    atr_raw.columns = atr_raw.columns.get_level_values(-1)
atr_raw = atr_raw.reindex(columns=close.columns)

ATRTrailing = vbt.IndicatorFactory(
    input_names=['close', 'atr'],
    param_names=['atr_multiplier'],
    output_names=['atr_trailing'],
).from_apply_func(atr_trailing_nb)

atr_sl  = ATRTrailing.run(close, atr_raw, atr_multiplier=ATR_MULT)
exits   = close.vbt.crossed_below(atr_sl.atr_trailing)

# time stop: exit after MAX_HOLD_DAYS bars in the trade
if MAX_HOLD_DAYS > 0:
    in_trade  = entries.shift(1, fill_value=False).cumsum()
    entry_bar = entries.cumsum()
    bars_held = in_trade - entry_bar.where(entries, other=np.nan).ffill().fillna(0)
    time_stop = (bars_held >= MAX_HOLD_DAYS) & ~entries
    exits     = exits | time_stop

# initial hard stop: swing-low relative distance  →  passed as sl_stop
lowest_low = _MIN.run(low, timeperiod=SWING_LOW_LB).real
if isinstance(lowest_low.columns, pd.MultiIndex):
    lowest_low.columns = lowest_low.columns.get_level_values(-1)
lowest_low = lowest_low.reindex(columns=close.columns)
sl_stop = ((close - lowest_low) / close).clip(lower=0)

# ── 4. Portfolio ──────────────────────────────────────────────────────────────
pf = vbt.Portfolio.from_signals(
    close    = close,
    entries  = entries,
    exits    = exits,
    sl_stop  = sl_stop,
    freq     = '1d',
    group_by = close.columns,        # one group per symbol
)

print(f'Entry signals : {entries.sum().sum():,}')
print(f'Exit signals  : {exits.sum().sum():,}')

# ── 5. Stats ──────────────────────────────────────────────────────────────────
all_metrics = list(pf.metrics.keys())
skip = {}
pf.stats(metrics=[m for m in all_metrics if m not in skip])


Entry signals : 2,418
Exit signals  : 11,152


Start                                  2016-06-20 00:00:00
End                                    2026-06-19 00:00:00
Period                                  2501 days 00:00:00
Start Value                                          100.0
End Value                                        115.02012
Total Return [%]                                  15.02012
Benchmark Return [%]                            268.735983
Max Gross Exposure [%]                               100.0
Total Fees Paid                                        0.0
Max Drawdown [%]                                 21.555984
Max Drawdown Duration         1401 days 15:16:21.818181824
Total Trades                                      7.242424
Total Closed Trades                               7.181818
Total Open Trades                                 0.060606
Open Trade PnL                                    0.083176
Win Rate [%]                                      39.25308
Best Trade [%]                                   24.5762